In [1]:
#import

import random
import cv2
import json
import copy
import torch
import gc
import os
import librosa
import pickle
import timm
import mne

import numpy as np
import albumentations as A
import pandas as pd
from tqdm import tqdm
from scipy.signal import butter, lfilter


import torch
import torchaudio
import torch.nn as nn
from torch.utils.data import DataLoader


In [2]:
CFG={
    'batch_size':32,
    'num_worker':4,
    'data':'/kaggle/input/hms-harmful-brain-activity-classification/test.csv',
    'weights_eeg_raw':'/kaggle/input/hms-2nd-place-solution/pytorch/hms-eeg_raw/1',
}

In [3]:
CFG['weights_eeg_raw']=[os.path.join(CFG['weights_eeg_raw'],x) for x in sorted(os.listdir(CFG['weights_eeg_raw']))]
CFG

{'batch_size': 32,
 'num_worker': 4,
 'data': '/kaggle/input/hms-harmful-brain-activity-classification/test.csv',
 'weights_eeg_raw': ['/kaggle/input/hms-2nd-place-solution/pytorch/hms-eeg_raw/1/avg_fold0.pth',
  '/kaggle/input/hms-2nd-place-solution/pytorch/hms-eeg_raw/1/avg_fold1.pth',
  '/kaggle/input/hms-2nd-place-solution/pytorch/hms-eeg_raw/1/avg_fold2.pth',
  '/kaggle/input/hms-2nd-place-solution/pytorch/hms-eeg_raw/1/avg_fold3.pth',
  '/kaggle/input/hms-2nd-place-solution/pytorch/hms-eeg_raw/1/avg_fold4.pth',
  '/kaggle/input/hms-2nd-place-solution/pytorch/hms-eeg_raw/1/avg_fold5.pth',
  '/kaggle/input/hms-2nd-place-solution/pytorch/hms-eeg_raw/1/avg_fold6.pth',
  '/kaggle/input/hms-2nd-place-solution/pytorch/hms-eeg_raw/1/avg_fold7.pth',
  '/kaggle/input/hms-2nd-place-solution/pytorch/hms-eeg_raw/1/avg_fold8.pth',
  '/kaggle/input/hms-2nd-place-solution/pytorch/hms-eeg_raw/1/avg_fold9.pth']}

In [4]:
#dataiter
class AlaskaDataIter():
    def __init__(self, df,
                 training_flag=False,shuffle=False,
                 use_spec=False,
                 use_eeg=False,
                 use_mix=False,
                 ll=0,rr=20,
                 flip=False,
                 use_mne_filter=True):
        
        self.flip_eeg=flip
        self.ll=ll
        self.rr=rr
        
        print(self.ll,self.rr, 'with mne filter:', use_mne_filter)
        
        
        self.training_flag = training_flag
        self.shuffle = shuffle

        self.raw_data_set_size = None     ##decided by self.parse_file


        self.df=df
        

        TARS = {'Seizure': 0, 'LPD': 1, 'GPD': 2, 'LRDA': 3, 'GRDA': 4, 'Other': 5}
        self.TARS2 = {x: y for y, x in TARS.items()}


        self.eeg_nms=['Fp1', 'F3', 'C3', 'P3', 'F7', 'T3', 'T5', 'O1', 'Fz', 'Cz', 'Pz',
       'Fp2', 'F4', 'C4', 'P4', 'F8', 'T4', 'T6', 'O2', 'EKG']

        self.LL = ['Fp1', 'F7', 'T3', 'T5', 'O1']

        self.RL = ['Fp2', 'F8', 'T4', 'T6', 'O2']

        self.LP = ['Fp1', 'F3', 'C3', 'P3', 'O1']

        self.RP = ['Fp2', 'F4', 'C4', 'P4', 'O2']

        self.mid = ['Fz', 'Cz', 'Pz']
        self.leads_dict = {value: index for index, value in enumerate(self.eeg_nms)}

        self.use_eeg = use_eeg
        self.use_spec = use_spec
        self.use_mix = use_mix
        self.use_mne_filter=use_mne_filter
        
        
        
    def __getitem__(self, item):

        return self.single_map_func(self.df.iloc[item], self.training_flag)

    def __len__(self):

        return len(self.df)

    def brain_lead(self, waves):
        waves = copy.deepcopy(waves)
        brain_leads = [self.LL, self.RL, self.LP, self.RP]

        leads = []

        for chain in brain_leads:
            for i in range(len(chain) - 1):
                tmp_lead = waves[self.leads_dict[chain[i]]] - waves[self.leads_dict[chain[i + 1]]]
                leads.append(tmp_lead)

        data = np.concatenate([leads], axis=0)
        
        return data
    def mirror_spec(self, data):

        # index_choice = [[0, 1, 3, 2], [0, 1, 2, 3], [1, 0, 2, 3], [1, 0, 3, 2]]
        indx = [1, 0, 3, 2]
        return data[..., indx]

    def mirror_eeg(self, data):

        indx1 = [0, 1, 2, 3, 4, 5, 6, 7]
        indx2 = [11, 12, 13, 14, 15, 16, 17, 18]

        data[indx1, ...], data[indx2, ...] = data[indx2, ...], data[indx1, ...]

        return data
    def butter_bandpass(self,lowcut, highcut, fs, order=5):
        return butter(order, [lowcut, highcut], fs=fs, btype="band")

    def butter_bandpass_filter(self,data, lowcut, highcut, fs, order=5):
        b, a = self.butter_bandpass(lowcut, highcut, fs, order=order)
        y = lfilter(b, a, data)
        return y
    def get_eeg(self, dp, is_training,flip=False):
        eeg_path = '/kaggle/input/hms-harmful-brain-activity-classification/test_eegs/%s.parquet' % (dp['eeg_id'])
        eeg = pd.read_parquet(eeg_path)

        
        offset = 0
        eeg = eeg.iloc[int(offset * 200):int(offset * 200) + 10000]

        waves = eeg.values

        waves = np.transpose(waves, axes=[1, 0])

        for i in range(waves.shape[0]):
            m = np.nanmean(waves[i])
            if np.isnan(waves[i]).mean() < 1:
                waves[i] = np.nan_to_num(waves[i], nan=m)
            else:
                waves[i] = 0
        
        if flip:
            waves=self.mirror_eeg(waves)
        waves = self.brain_lead(waves)
        waves = np.array(waves, dtype=np.float64)

        waves = np.clip(waves, -1024, 1024)
        if self.use_mne_filter:
            waves = mne.filter.filter_data(waves, 200, self.ll, self.rr, verbose=False)
        else:
            waves = self.butter_bandpass_filter(waves,0.5,20,200,2)
        #


        return waves
          
    
    def single_map_func(self, dp, is_training):
        
        ####customed here
        data=self.get_eeg(dp,is_training,self.flip_eeg)
        return data.astype(np.float32)

In [5]:
class Net1d(nn.Module):
    def __init__(self,):
        super(Net1d, self).__init__()
        self.model=timm.create_model('efficientnet_b5', pretrained=False, in_chans=3)
        self.pool=nn.AdaptiveAvgPool2d(1)
        self.fc=nn.Linear(2048,out_features=6,bias=True)
        self.dropout=nn.Dropout(p=0.5)

    def extract_features(self, x):
        feature1=self.model.forward_features(x)
        return feature1
    def forward(self, x):
        
        bs = x.size(0)
        reshaped_tensor = x.view(bs,16,1000, 10)
        reshaped_and_permuted_tensor = reshaped_tensor.permute(0,1,3,2)
        reshaped_and_permuted_tensor= reshaped_and_permuted_tensor.reshape(bs,16*10,1000)
        x=torch.unsqueeze(reshaped_and_permuted_tensor,dim=1)
        x=torch.cat([x,x,x],dim=1)
        bs=x.size(0)

        x = self.extract_features(x)

        # print(x.size())
        x = self.pool(x)
        x = x.view(bs, -1)
        
        x =self.dropout(x)
        x = self.fc(x)
        x =torch.softmax(x,-1)
        
        return x

In [6]:
test_df=pd.read_csv(CFG['data'])

test_df.head(5)

,spectrogram_id,eeg_id,patient_id
0,853520,3911565283,6885


In [7]:
def inference_function(test_loader, model, device,double_input=False):
    model.eval()
    
    prediction_dict = {}
    preds = []
    with tqdm(test_loader, unit="test_batch", desc='Inference') as tqdm_test_loader:
        for step, X in enumerate(tqdm_test_loader):
            
            X = X.to(device)

            batch_size = X.size(0)
            with torch.no_grad():
                y_preds = model(X)

            preds.append(y_preds.to('cpu').numpy()) 
                
    prediction_dict["predictions"] = np.concatenate(preds) 
    return prediction_dict

In [8]:
def run_weight_eeg_raw():
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    print('infer with run_weight_eeg_raw')
    predictions=[]
    for model_weight in CFG['weights_eeg_raw']:
        test_dataset = AlaskaDataIter(test_df, training_flag=False, shuffle=False,use_eeg=True,ll=0.5,rr=20)
        test_loader = DataLoader(test_dataset,
                         CFG['batch_size'],
                         num_workers=CFG['num_worker'],
                         shuffle=False)

        model = Net1d()
        state_dict = torch.load(model_weight, map_location=device)

        model.load_state_dict(state_dict,strict=True)
        model.to(device)
        prediction_dict = inference_function(test_loader, model, device)
        predictions.append(prediction_dict["predictions"])


        test_dataset = AlaskaDataIter(test_df, training_flag=False, shuffle=False,use_eeg=True,flip=True,ll=0.5,rr=20)
        test_loader = DataLoader(test_dataset,
                         CFG['batch_size'],
                         num_workers=CFG['num_worker'],
                         shuffle=False)

        model = Net1d()
        state_dict = torch.load(model_weight, map_location=device)

        model.load_state_dict(state_dict,strict=True)
        model.to(device)
        prediction_dict = inference_function(test_loader, model, device)
        predictions.append(prediction_dict["predictions"])

        torch.cuda.empty_cache()
        gc.collect()
    predictions = np.array(predictions)
    predictions = np.mean(predictions, axis=0)
    
    return predictions

In [9]:
predictions = []

# #***0.23 best version29, add 1000 val data
ans=run_weight_eeg_raw()
predictions.append(ans) 


predictions=np.array(predictions)
predictions=np.mean(predictions,axis=0)


infer with run_weight_eeg_raw
0.5 20 with mne filter: True


Inference: 100%|██████████| 1/1 [00:01<00:00,  1.42s/test_batch]


0.5 20 with mne filter: True


Inference: 100%|██████████| 1/1 [00:00<00:00,  2.50test_batch/s]


0.5 20 with mne filter: True


...

Inference: 100%|██████████| 1/1 [00:00<00:00,  2.43test_batch/s]


0.5 20 with mne filter: True


Inference: 100%|██████████| 1/1 [00:00<00:00,  2.25test_batch/s]


0.5 20 with mne filter: True


Inference: 100%|██████████| 1/1 [00:00<00:00,  2.28test_batch/s]


In [10]:
TARGETS = ['seizure_vote', 'lpd_vote', 'gpd_vote', 'lrda_vote', 'grda_vote', 'other_vote']
sub = pd.DataFrame({'eeg_id': test_df.eeg_id.values})
sub[TARGETS] = predictions
sub.to_csv('submission.csv',index=False)
print(f'Submissionn shape: {sub.shape}')
sub.head()

Submissionn shape: (1, 7)


,eeg_id,seizure_vote,lpd_vote,gpd_vote,lrda_vote,grda_vote,other_vote
0,3911565283,0.046691,0.022814,0.00017,0.340036,0.012714,0.577575
